# Subterranean Boulder & Obstacle Segmentation Model Training

This notebook documents the full training pipeline, hyperparameter tuning, convergence analysis, and edge optimization for the **Boulder & Rock Obstacle Instance Segmentation Model** deployed on the ELIOS Search & Rescue UAV.

### Operational Context:
In subterranean mines and collapsed structural voids, dislodged rock masses and falling boulders pose critical collision risks for autonomous micro-UAVs and block rescue ingress routes. This model provides real-time pixel-level polygon segmentation masks of rock obstructions to calculate corridor traversability and distance metrics.

- **Architecture:** YOLOv8 / YOLO11 Nano-Segmentation (`yolo-seg`)
- **Dataset:** `akahukas/rock-segmentation-dataset` (Tampere University Autonomous Mobile Machines Group)
- **Target Class:** `rock` (0)
- **Deployment Platform:** Embedded ARM Cortex-A76 (Raspberry Pi 5 / Jetson Orin Nano)


## 1. Environment Setup & Hardware Acceleration

Verify Python environment, PyTorch installation, and CUDA hardware acceleration.


In [ ]:
import sys
import os
from pathlib import Path
import torch

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
gpu_available = torch.cuda.is_available()
print(f"CUDA Hardware Acceleration: {gpu_available}")
if gpu_available:
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Allocated: {torch.cuda.memory_allocated(0)/(1024**2):.2f} MB")
else:
    print("Running in CPU mode. For training, a T4/V100/A100 GPU is recommended.")

try:
    import ultralytics
    print(f"Ultralytics Version: {ultralytics.__version__}")
except ImportError:
    print("Installing Ultralytics...")
    !pip install ultralytics pandas matplotlib seaborn opencv-python --quiet
    import ultralytics
    print(f"Ultralytics Installed: {ultralytics.__version__}")


## 2. Dataset Specification & Roboflow/HuggingFace Integration

The model is trained on the dedicated **Rock Segmentation Dataset** comprising multi-angle RGB captures of fractured rock, subterranean boulders, and gravel formations with polygon segmentation ground truth.

Dataset Structure:
```
dataset/
├── train/
│   ├── images/
│   └── labels/
├── valid/
│   ├── images/
│   └── labels/
└── data.yaml
```


In [ ]:
import yaml

# Dataset configuration definition
data_yaml_content = {
    'path': '../dataset',
    'train': 'train/images',
    'val': 'valid/images',
    'names': {
        0: 'rock'
    }
}

data_yaml_path = Path("rock_data.yaml")
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False)

print(f"Created dataset configuration file: {data_yaml_path.resolve()}")
print(yaml.dump(data_yaml_content, default_flow_style=False))


## 3. Model Architecture Initialization & Training Configuration

We initialize from a pre-trained `yolo11n-seg.pt` (or `yolo26n-seg.pt`) checkpoint as the foundational spatial feature extractor. We configure hyperparameters for edge robotics:
- **Image Size:** 640x640 during training, scaled to 320x320 for embedded deployment.
- **Batch Size:** 32 with dynamic gradient accumulation.
- **Loss Functions:**
  - Box Loss: Complete IoU (CIoU) Loss
  - Mask Segmentation Loss: Binary Cross-Entropy (BCE) Loss over prototype masks
  - Distribution Focal Loss (DFL) for precise bounding box regression
- **Augmentation:** Multi-scale jitter, horizontal flip (p=0.5), HSV color jitter (to handle mine lamp variations).


In [ ]:
from ultralytics import YOLO

# Initialize instance segmentation model architecture
model = YOLO('yolo11n-seg.pt')

# Training execution parameters
training_params = {
    'data': str(data_yaml_path),
    'epochs': 500,
    'imgsz': 640,
    'batch': 32,
    'workers': 8,
    'optimizer': 'SGD',
    'lr0': 0.01,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3.0,
    'close_mosaic': 10,
    'save': True,
    'project': 'runs/segment',
    'name': 'boulder_edge_seg',
    'exist_ok': True
}

print("Training Pipeline Configuration:")
for k, v in training_params.items():
    print(f"  {k}: {v}")

# Execute training (uncomment when running in full training environment)
# results = model.train(**training_params)


## 4. Empirical Training Convergence & Loss Analysis

We inspect the actual empirical training telemetry recorded over 514 epochs. We parse `BOULDER/results.csv` to analyze convergence across bounding box loss, mask segmentation loss, class loss, distribution focal loss, and mask mAP metrics.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    plt.style.use('default')

# Path to the actual training results CSV
csv_candidates = [
    Path("../BOULDER/results.csv"),
    Path("BOULDER/results.csv"),
    Path("../runs/segment/boulder_edge_seg/results.csv")
]

results_file = next((p for p in csv_candidates if p.exists()), None)

if results_file:
    print(f"Loading empirical training logs from: {results_file.resolve()}")
    df = pd.read_csv(results_file)
    # Strip whitespace from column headers
    df.columns = df.columns.str.strip()
    
    print(f"Total Epochs Completed: {len(df)}")
    print(f"Columns Recorded: {list(df.columns)}")
    
    # Display final converged metrics
    final_epoch = df.iloc[-1]
    print("\n=== Final Converged Validation Metrics ===")
    print(f"Box Precision:  {final_epoch.get('metrics/precision(B)', 0):.4f}")
    print(f"Box Recall:     {final_epoch.get('metrics/recall(B)', 0):.4f}")
    print(f"Box mAP@50:     {final_epoch.get('metrics/mAP50(B)', 0):.4f}")
    print(f"Box mAP@50-95:  {final_epoch.get('metrics/mAP50-95(B)', 0):.4f}")
    print(f"Mask Precision: {final_epoch.get('metrics/precision(M)', 0):.4f}")
    print(f"Mask Recall:    {final_epoch.get('metrics/recall(M)', 0):.4f}")
    print(f"Mask mAP@50:    {final_epoch.get('metrics/mAP50(M)', 0):.4f}")
    print(f"Mask mAP@50-95: {final_epoch.get('metrics/mAP50-95(M)', 0):.4f}")
    
    # Plotting Loss and Metric Trajectories
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    sns.set_theme(style="whitegrid")
    
    # 1. Bounding Box & Segmentation Training Losses
    axes[0, 0].plot(df['epoch'], df['train/box_loss'], label='Train Box Loss', color='#1f77b4', lw=1.8)
    axes[0, 0].plot(df['epoch'], df['val/box_loss'], label='Val Box Loss', color='#ff7f0e', lw=1.8, linestyle='--')
    axes[0, 0].set_title('Bounding Box Regression Loss (CIoU)', fontsize=12, fontweight='bold')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].legend()
    
    # 2. Mask Segmentation Loss
    axes[0, 1].plot(df['epoch'], df['train/seg_loss'], label='Train Mask Seg Loss', color='#2ca02c', lw=1.8)
    axes[0, 1].plot(df['epoch'], df['val/seg_loss'], label='Val Mask Seg Loss', color='#d62728', lw=1.8, linestyle='--')
    axes[0, 1].set_title('Instance Mask Loss (Proto-BCE)', fontsize=12, fontweight='bold')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].legend()
    
    # 3. Distribution Focal Loss (DFL)
    axes[0, 2].plot(df['epoch'], df['train/dfl_loss'], label='Train DFL Loss', color='#9467bd', lw=1.8)
    axes[0, 2].plot(df['epoch'], df['val/dfl_loss'], label='Val DFL Loss', color='#8c564b', lw=1.8, linestyle='--')
    axes[0, 2].set_title('Distribution Focal Loss (DFL)', fontsize=12, fontweight='bold')
    axes[0, 2].set_xlabel('Epoch')
    axes[0, 2].legend()
    
    # 4. Box Detection Accuracy (mAP@50 & mAP@50-95)
    axes[1, 0].plot(df['epoch'], df['metrics/mAP50(B)'], label='Box mAP@50', color='#007acc', lw=2)
    axes[1, 0].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='Box mAP@50-95', color='#ff6600', lw=2)
    axes[1, 0].set_title('Bounding Box mAP Convergence', fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].legend()
    
    # 5. Mask Segmentation Accuracy (mAP@50 & mAP@50-95)
    axes[1, 1].plot(df['epoch'], df['metrics/mAP50(M)'], label='Mask mAP@50', color='#2ca02c', lw=2)
    axes[1, 1].plot(df['epoch'], df['metrics/mAP50-95(M)'], label='Mask mAP@50-95', color='#d62728', lw=2)
    axes[1, 1].set_title('Mask Segmentation mAP Convergence', fontsize=12, fontweight='bold')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].legend()
    
    # 6. Precision-Recall Curve over Training
    axes[1, 2].plot(df['epoch'], df['metrics/precision(M)'], label='Mask Precision', color='#17becf', lw=1.8)
    axes[1, 2].plot(df['epoch'], df['metrics/recall(M)'], label='Mask Recall', color='#bcbd22', lw=1.8)
    axes[1, 2].set_title('Mask Precision & Recall Trajectory', fontsize=12, fontweight='bold')
    axes[1, 2].set_xlabel('Epoch')
    axes[1, 2].legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("Results CSV not found. Please ensure BOULDER/results.csv is in the repository.")


## 5. Ground Truth vs Validation Prediction Inspection

Visual audit of model segmentation fidelity on unseen validation batches to ensure zero contour leaking on uneven rock textures.


In [ ]:
from PIL import Image

val_labels_path = Path("../BOULDER/val_batch1_labels.jpg")
val_preds_path = Path("../BOULDER/val_batch1_pred.jpg")

if not val_labels_path.exists():
    val_labels_path = Path("BOULDER/val_batch1_labels.jpg")
    val_preds_path = Path("BOULDER/val_batch1_pred.jpg")

if val_labels_path.exists() and val_preds_path.exists():
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    img_labels = Image.open(val_labels_path)
    axes[0].imshow(img_labels)
    axes[0].set_title("Ground Truth Validation Batch Annotations", fontsize=13, fontweight='bold')
    axes[0].axis('off')
    
    img_preds = Image.open(val_preds_path)
    axes[1].imshow(img_preds)
    axes[1].set_title("Trained Model Predictions (Box + Segmentation Mask)", fontsize=13, fontweight='bold')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("Validation sample images not found in BOULDER directory.")


## 6. UAV Flight Corridor Obstacle Analysis & Severity Logic

Here we simulate the exact runtime segmentation processor implemented in `src/perception/boulder_detector.py` and `src/perception/segmentation_detector.py`.

The processor:
1. Performs instance segmentation on incoming video frames at `imgsz=320`
2. Computes the **mask area ratio** relative to the total frame ($Area_{\text{mask}} / Area_{\text{frame}}$)
3. Evaluates **Central Corridor Overlap** (whether the rock mass blocks the drone's primary heading)
4. Assesses **Touch Bottom** (indicating floor blockage immediately below the drone)
5. Assigns an obstruction severity rating: `low`, `medium`, or `critical`.


In [ ]:
import numpy as np
import cv2

def evaluate_boulder_hazard(mask, frame_shape):
    """
    Calculates operational risk metrics for autonomous drone path planning.
    """
    height, width = frame_shape[:2]
    total_area = height * width
    mask_pixels = np.count_nonzero(mask)
    area_ratio = mask_pixels / total_area
    
    # 1. Central corridor evaluation (middle 40% horizontal slice)
    corridor_x_min = int(width * 0.30)
    corridor_x_max = int(width * 0.70)
    central_slice = mask[:, corridor_x_min:corridor_x_max]
    central_overlap = np.any(central_slice > 0)
    
    # 2. Bottom touch evaluation (bottom 10% of the image)
    bottom_slice = mask[int(height * 0.90):, :]
    touches_bottom = np.any(bottom_slice > 0)
    
    # 3. Severity classification
    if area_ratio > 0.30 and central_overlap:
        severity = "critical"
    elif area_ratio > 0.15 or (central_overlap and touches_bottom):
        severity = "medium"
    else:
        severity = "low"
        
    return {
        "mask_area_ratio": area_ratio,
        "central_corridor_overlap": central_overlap,
        "touches_bottom": touches_bottom,
        "severity": severity
    }

# Synthetic validation test of severity logic
test_mask = np.zeros((320, 320), dtype=np.uint8)
cv2.circle(test_mask, (160, 200), 75, 255, -1)  # Simulated boulder obstruction

hazard_metrics = evaluate_boulder_hazard(test_mask, (320, 320))
print("Corridor Blockage Assessment Test:")
for k, v in hazard_metrics.items():
    print(f"  {k}: {v}")


## 7. Model Export for Low-Power Embedded Edge Deployment

For onboard inference on the drone's Raspberry Pi 5 / Jetson Orin Nano, we export the trained weights to **ONNX** and **TorchScript** formats with half-precision (FP16) or INT8 quantization for optimal throughput.


In [ ]:
# Path to the trained checkpoint
model_path = Path("../models/hazards/boulder.pt")
if not model_path.exists():
    model_path = Path("models/hazards/boulder.pt")

if model_path.exists():
    print(f"Exporting model from: {model_path.resolve()}")
    edge_model = YOLO(str(model_path))
    
    # Export to ONNX format (ideal for ONNXRuntime / OpenVINO on edge CPU)
    try:
        onnx_file = edge_model.export(format='onnx', imgsz=320, half=False, simplify=True)
        print(f"Successfully exported to ONNX: {onnx_file}")
    except Exception as e:
        print(f"ONNX export preview: {e}")
else:
    print(f"Checkpoint not found at {model_path}. Provide trained weights to export.")
